# 05. Analytics, Temporal Reasoning & Derived Calculations

Covers **Attributes 14, 15, 19, 20 & Coding Sub-Agent**:
- Standardized formatting (markdown tables, USD/hL/%)
- Temporal reasoning (historical, current YTD 2026, YoY)
- Enterprise-wide superlative comparative analysis (poor/best years)
- Multi-entity, multi-KPI, and channel comparisons
- Sandboxed derived calculations via Coding Sub-Agent (CAGR, projections)

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if "high_level" in str(pathlib.Path.cwd()) or "capabilities" in str(pathlib.Path.cwd()) else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.orchestrator import Orchestrator
from src.llm_client import MockLLMClient
from src.tools.sql_tool import run_query, validate_sql
from src.tools.retrieval_tool import get_index
from src.tools.code_tool import run_code
from src.formatting import format_value, rows_to_markdown_table

print("AB InBev Enterprise Q&A Agent Pipeline Loaded.")

AB InBev Enterprise Q&A Agent Pipeline Loaded.

### 1. Comparative Superlative Query & Multi-Year Performance

In [2]:
orch = Orchestrator(llm_router=MockLLMClient(), llm_worker=MockLLMClient())
q = "in year did the AB inBev performed poor comparatively"
r = orch.handle_turn(q)
print(f"Query: {q}\nIntent: {r.intent}\nSQL:\n{r.sql_used}\n\nAnswer:\n{r.answer}")

Query: in year did the AB inBev performed poor comparatively
Intent: comparison
SQL:
SELECT year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(gross_margin_pct) AS gross_margin_pct, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi GROUP BY year ORDER BY year LIMIT 500

Answer:
Comparatively across historical reporting, **2023 was AB InBev's lowest-performing full year**, recording the lowest net revenue ($140,971,636), lowest volume (1,790,001 hL), and lowest average market share (18.0%).

| Year | Net Revenue (USD) | Volume (hL) | Gross Margin (%) | Market Share (%) |
| --- | --- | --- | --- | --- |
| 2023 | $140,971,636 | 1,790,000.8 hL | 52.0% | 18.0% |
| 2024 | $156,046,898 | 1,914,265.4 hL | 52.0% | 18.3% |
| 2025 | $172,003,014 | 2,045,475.4 hL | 52.0% | 18.6% |
| 2026 | $125,652,879 | 1,455,549.9 hL | 52.0% | 18.8% |


### Supporting Evidence & Percentage Variances:
- **2023 vs. 2024 Performance**:
  - **Net Revenue**: 2023 was **9.66% lower

### 2. Multi-Brand & Multi-KPI Comparison

In [3]:
q = "Compare Budweiser and Corona in the United States in 2025"
r = orch.handle_turn(q)
print(f"Query: {q}\nSQL: {r.sql_used}\n\nAnswer:\n{r.answer}")

Query: Compare Budweiser and Corona in the United States in 2025
SQL: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand IN ('Corona', 'Budweiser') AND country='United States' AND year=2025 GROUP BY brand, country, year LIMIT 500

Answer:
In 2025, Budweiser in United States recorded net revenue of **$3,483,878** and volume of **57,558.2 hL**.

| Brand | Country | Year | Net Revenue (USD) | Volume (hL) | Market Share (%) |
| --- | --- | --- | --- | --- | --- |
| Budweiser | United States | 2025 | $3,483,878 | 57,558.2 hL | 14.8% |
| Corona | United States | 2025 | $9,230,264 | 91,438.5 hL | 15.3% |


The figures highlight healthy commercial execution across AB InBev's core markets and channels.

### 3. Sandboxed Coding Sub-Agent Execution (CAGR & Projections)

In [4]:
queries = [
    "Calculate CAGR if revenue grew from 140.97 to 172.00 over 2 years",
    "If Corona grew by 8% per year for 3 years calculate the projection from 172"
]
for q in queries:
    r = orch.handle_turn(q)
    print(f">> {q}")
    print(f"Sub-Agents: {r.sub_agents_used}")
    print(f"Code Executed:\n{r.intermediate_steps.get('code_used')}\n")

>> Calculate CAGR if revenue grew from 140.97 to 172.00 over 2 years
Sub-Agents: ['structured', 'coding']
Code Executed:
result = round((1 + 0.06) ** 5, 2)

>> If Corona grew by 8% per year for 3 years calculate the projection from 172
Sub-Agents: ['structured', 'coding']
Code Executed:
result = round((1 + 0.06) ** 5, 2)
